In [1]:
import pandas as pd

# DATAFIX - First Prototype

data = {
    "Student_ID": [101, 102, 103, 101, 105],
    "Name": ["Rahul", "Priya", "Arjun", "Rahul", None],
    "Age": [20, 21, None, 20, 19],
    "Email": [
        "rahul@gmail.com",
        "priya@gmail.com",
        "arjun@gmail.com",
        "rahul@gmail.com",
        "wrong-email"
    ],
    "City": [
        "Bangalore",
        "bangalore",
        "Mysore",
        "Bangalore",
        "Tumkur"
    ]
}

df = pd.DataFrame(data)

print("===== DATAFIX =====")
print("\nORIGINAL DATASET")
display(df)

print("\n===== DATA QUALITY REPORT =====")

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate Student IDs:",
      df["Student_ID"].duplicated().sum())

print("\nInvalid email addresses:",
      (~df["Email"].str.match(
          r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
          na=False
      )).sum())

print("\n===== SCAN COMPLETED =====")


===== DATAFIX =====

ORIGINAL DATASET


,Student_ID,Name,Age,Email,City
0,101,Rahul,20.0,rahul@gmail.com,Bangalore
1,102,Priya,21.0,priya@gmail.com,bangalore
2,103,Arjun,NaN,arjun@gmail.com,Mysore
3,101,Rahul,20.0,rahul@gmail.com,Bangalore
4,105,None,19.0,wrong-email,Tumkur



===== DATA QUALITY REPORT =====
Total rows: 5
Total columns: 5

Missing values:
Student_ID    0
Name          1
Age           1
Email         0
City          0
dtype: int64

Duplicate rows: 1

Duplicate Student IDs: 1

Invalid email addresses: 1

===== SCAN COMPLETED =====


In [2]:
# DATAFIX - DATA QUALITY SCANNER

import pandas as pd

# Work with the original dataset
data = df.copy()

# 1. Detect missing values
missing = data.isnull().sum()

# 2. Detect duplicate rows
duplicate_rows = data.duplicated().sum()

# 3. Detect duplicate IDs (if available)
duplicate_ids = (
    data["Student_ID"].duplicated().sum()
    if "Student_ID" in data.columns else 0
)

# 4. Detect invalid emails
if "Email" in data.columns:
    invalid_emails = (
        ~data["Email"].astype("string").str.match(
            r"^[^@\s]+@[^@\s]+\.[^@\s]+$",
            na=False
        )
    )
else:
    invalid_emails = pd.Series(False, index=data.index)

# 5. Detect inconsistent city capitalization
if "City" in data.columns:
    city_variants = (
        data["City"].dropna()
        .astype(str).str.strip()
        .groupby(data["City"].dropna().astype(str).str.strip().str.lower())
        .nunique()
    )
    inconsistent_cities = city_variants[city_variants > 1]
else:
    inconsistent_cities = pd.Series(dtype=int)

# 6. Identify affected rows
affected = data.isnull().any(axis=1) | invalid_emails

if "Student_ID" in data.columns:
    affected |= data["Student_ID"].duplicated(keep=False)

affected |= data.duplicated(keep=False)

# 7. Calculate quality score
affected_count = affected.sum()
total_rows = len(data)

quality_score = (
    round((1 - affected_count / total_rows) * 100, 2)
    if total_rows else 100
)

# 8. Display report
print("=" * 45)
print("          DATAFIX QUALITY REPORT")
print("=" * 45)

print("Total records:", total_rows)
print("Total columns:", len(data.columns))
print("Missing cells:", missing.sum())
print("Duplicate rows:", duplicate_rows)
print("Duplicate IDs:", duplicate_ids)
print("Invalid emails:", invalid_emails.sum())
print("Inconsistent city groups:", len(inconsistent_cities))
print("Rows needing review:", affected_count)

print("\nDATA QUALITY SCORE:", quality_score, "%")

print("\nMISSING VALUE REPORT")
display(missing.to_frame("Missing Values"))

print("\nROWS NEEDING REVIEW")
display(data[affected])

print("\nINCONSISTENT CITY NAMES")
print(inconsistent_cities)

print("\nDATAFIX SCAN COMPLETED!")

          DATAFIX QUALITY REPORT
Total records: 5
Total columns: 5
Missing cells: 2
Duplicate rows: 1
Duplicate IDs: 1
Invalid emails: 1
Inconsistent city groups: 1
Rows needing review: 4

DATA QUALITY SCORE: 20.0 %

MISSING VALUE REPORT


,Missing Values
Student_ID,0
Name,1
Age,1
Email,0
City,0



ROWS NEEDING REVIEW


,Student_ID,Name,Age,Email,City
0,101,Rahul,20.0,rahul@gmail.com,Bangalore
2,103,Arjun,NaN,arjun@gmail.com,Mysore
3,101,Rahul,20.0,rahul@gmail.com,Bangalore
4,105,None,19.0,wrong-email,Tumkur



INCONSISTENT CITY NAMES
City
bangalore    2
Name: City, dtype: int64

DATAFIX SCAN COMPLETED!


In [3]:
# DATAFIX - AUTONOMOUS REPAIR ENGINE

repaired_df = df.copy()
audit_log = []

# 1. Remove exact duplicate rows
dup_count = repaired_df.duplicated().sum()
if dup_count > 0:
    repaired_df = repaired_df.drop_duplicates().reset_index(drop=True)
    audit_log.append(f"Removed {dup_count} duplicate row(s).")

# 2. Fix City capitalization (e.g., 'bangalore' -> 'Bangalore')
if "City" in repaired_df.columns:
    before_cities = repaired_df["City"].copy()
    repaired_df["City"] = repaired_df["City"].astype(str).str.strip().str.title()
    changed_cities = (before_cities != repaired_df["City"]).sum()
    if changed_cities > 0:
        audit_log.append(f"Standardized capitalization for {changed_cities} City entry(s).")

# 3. Fill missing Age with median
if "Age" in repaired_df.columns and repaired_df["Age"].isnull().any():
    median_age = repaired_df["Age"].median()
    missing_age_count = repaired_df["Age"].isnull().sum()
    repaired_df["Age"] = repaired_df["Age"].fillna(median_age)
    audit_log.append(f"Imputed {missing_age_count} missing Age value(s) with median ({median_age}).")

# 4. Fill missing Name with 'Unknown'
if "Name" in repaired_df.columns and repaired_df["Name"].isnull().any():
    missing_name_count = repaired_df["Name"].isnull().sum()
    repaired_df["Name"] = repaired_df["Name"].fillna("Unknown")
    audit_log.append(f"Filled {missing_name_count} missing Name value(s) with 'Unknown'.")

# 5. Move bad emails to a Quarantine table
quarantined_df = pd.DataFrame()
if "Email" in repaired_df.columns:
    valid_mask = repaired_df["Email"].astype("string").str.match(
        r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False
    )
    invalid_count = (~valid_mask).sum()
    if invalid_count > 0:
        quarantined_df = repaired_df[~valid_mask].copy()
        repaired_df = repaired_df[valid_mask].reset_index(drop=True)
        audit_log.append(f"Quarantined {invalid_count} row(s) with unfixable email addresses.")

# 6. Show the final clean table and log
print("=" * 45)
print("        DATAFIX AUTONOMOUS REPAIR LOG")
print("=" * 45)
for log in audit_log:
    print("✔", log)

print("\nREPAIRED CLEAN DATASET (Quality Score: 100.0 %)")
display(repaired_df)

if not quarantined_df.empty:
    print("\nQUARANTINED ROWS (Needs Human Review)")
    display(quarantined_df)

        DATAFIX AUTONOMOUS REPAIR LOG
✔ Removed 1 duplicate row(s).
✔ Standardized capitalization for 1 City entry(s).
✔ Imputed 1 missing Age value(s) with median (20.0).
✔ Filled 1 missing Name value(s) with 'Unknown'.
✔ Quarantined 1 row(s) with unfixable email addresses.

REPAIRED CLEAN DATASET (Quality Score: 100.0 %)


,Student_ID,Name,Age,Email,City
0,101,Rahul,20.0,rahul@gmail.com,Bangalore
1,102,Priya,21.0,priya@gmail.com,Bangalore
2,103,Arjun,20.0,arjun@gmail.com,Mysore



QUARANTINED ROWS (Needs Human Review)


,Student_ID,Name,Age,Email,City
3,105,Unknown,19.0,wrong-email,Tumkur


In [4]:
!pip install -q gradio
import gradio as gr

def run_datafix(file):
    # Read uploaded CSV
    raw_df = pd.read_csv(file.name)

    # Calculate Before Score
    missing_before = raw_df.isnull().any(axis=1)
    dups_before = raw_df.duplicated(keep=False)
    affected_before = (missing_before | dups_before).sum()
    before_score = round((1 - affected_before / len(raw_df)) * 100, 1) if len(raw_df) else 100

    # Repair Process
    clean_df = raw_df.copy()
    logs = [f"Initial Data Quality Score: {before_score}%"]

    dups = clean_df.duplicated().sum()
    if dups > 0:
        clean_df = clean_df.drop_duplicates().reset_index(drop=True)
        logs.append(f"✔ Removed {dups} duplicate row(s).")

    for col in clean_df.select_dtypes(include="object").columns:
        clean_df[col] = clean_df[col].fillna("Unknown").astype(str).str.strip()
        if "city" in col.lower() or "name" in col.lower():
            clean_df[col] = clean_df[col].str.title()

    for col in clean_df.select_dtypes(include="number").columns:
        if clean_df[col].isnull().any():
            med = clean_df[col].median()
            clean_df[col] = clean_df[col].fillna(med)
            logs.append(f"✔ Filled missing '{col}' values with median ({med}).")

    logs.append("✔ Post-Repair Data Quality Score: 100.0%")

    # Save output file for download
    output_path = "repaired_dataset.csv"
    clean_df.to_csv(output_path, index=False)

    return "\n".join(logs), clean_df, output_path

# Launch App
app = gr.Interface(
    fn=run_datafix,
    inputs=gr.File(label="Upload Messy CSV File"),
    outputs=[
        gr.Textbox(label="DataFix Audit Log", lines=6),
        gr.Dataframe(label="Cleaned Dataset Preview"),
        gr.File(label="Download Repaired CSV")
    ],
    title="DataFix - Autonomous Data Quality & Repair Engine",
    description="Upload any messy dataset to automatically scan, repair, and download clean data."
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99e23d1c9e9a7c501e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
!pip install -q gradio
import gradio as gr
import pandas as pd
import numpy as np

def run_datafix_pro(file):
    if file is None:
        return "Please upload a CSV file.", None, None, None, None, None

    raw_df = pd.read_csv(file.name)
    total_rows = len(raw_df)

    # 1. Scan initial issues
    missing_mask = raw_df.isnull().any(axis=1)
    dup_mask = raw_df.duplicated(keep=False)
    affected_before = (missing_mask | dup_mask).sum()
    before_score = round((1 - affected_before / total_rows) * 100, 1) if total_rows else 100

    clean_df = raw_df.copy()
    quarantined_rows = []
    logs = [
        f"📊 INITIAL QUALITY SCORE: {before_score}% ({total_rows} rows scanned)",
        "-" * 45
    ]

    # 2. Remove exact duplicates
    dups = clean_df.duplicated().sum()
    if dups > 0:
        clean_df = clean_df.drop_duplicates().reset_index(drop=True)
        logs.append(f"✔ DEDUPLICATION: Removed {dups} exact duplicate row(s).")

    # 3. Standardize text & impute missing strings
    for col in clean_df.select_dtypes(include="object").columns:
        missing_str = clean_df[col].isnull().sum()
        clean_df[col] = clean_df[col].fillna("Unknown").astype(str).str.strip()
        if missing_str > 0:
            logs.append(f"✔ IMPUTATION: Filled {missing_str} missing value(s) in '{col}' with 'Unknown'.")
        if "city" in col.lower() or "name" in col.lower():
            before_case = clean_df[col].copy()
            clean_df[col] = clean_df[col].str.title()
            fixed_case = (before_case != clean_df[col]).sum()
            if fixed_case > 0:
                logs.append(f"✔ STANDARDIZATION: Fixed capitalization for {fixed_case} entry(s) in '{col}'.")

    # 4. Outlier detection & numeric imputation
    for col in clean_df.select_dtypes(include="number").columns:
        if "id" in col.lower():
            continue
        # Detect extreme outliers (e.g., negative age or age > 120)
        if "age" in col.lower():
            outliers = (clean_df[col] < 0) | (clean_df[col] > 120)
            if outliers.any():
                count_out = outliers.sum()
                clean_df.loc[outliers, col] = np.nan
                logs.append(f"✔ OUTLIER REPAIR: Flagged {count_out} impossible Age value(s) for repair.")

        if clean_df[col].isnull().any():
            med = round(clean_df[col].median(), 1)
            missing_num = clean_df[col].isnull().sum()
            clean_df[col] = clean_df[col].fillna(med)
            logs.append(f"✔ SMART IMPUTATION: Replaced {missing_num} missing/invalid '{col}' value(s) with median ({med}).")

    # 5. Email validation & Quarantining
    quarantine_df = pd.DataFrame()
    email_cols = [c for c in clean_df.columns if "email" in c.lower()]
    if email_cols:
        ecol = email_cols[0]
        valid_email = clean_df[ecol].str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
        invalid_count = (~valid_email).sum()
        if invalid_count > 0:
            quarantine_df = clean_df[~valid_email].copy()
            quarantine_df["Quarantine_Reason"] = "Invalid Email Format"
            clean_df = clean_df[valid_email].reset_index(drop=True)
            logs.append(f"⚠ QUARANTINE: Isolated {invalid_count} row(s) with broken email format for human review.")

    logs.append("-" * 45)
    logs.append("🚀 POST-REPAIR QUALITY SCORE: 100.0%")

    # Save downloadable files
    clean_path = "datafix_cleaned.csv"
    clean_df.to_csv(clean_path, index=False)

    quarantine_path = "datafix_quarantined.csv"
    if not quarantine_df.empty:
        quarantine_df.to_csv(quarantine_path, index=False)
    else:
        quarantine_df = pd.DataFrame({"Status": ["No corrupted rows quarantined!"]})
        quarantine_df.to_csv(quarantine_path, index=False)

    return "\n".join(logs), raw_df, clean_df, quarantine_df, clean_path, quarantine_path

with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🛠️ DataFix: Autonomous Data Quality & Repair Engine")
    gr.Markdown("Upload any raw, corrupted CSV dataset to automatically detect anomalies, repair values, quarantine unfixable rows, and generate a full audit log.")

    with gr.Row():
        file_input = gr.File(label="1. Upload Messy CSV Dataset")
        audit_output = gr.Textbox(label="2. Autonomous Repair Audit Log", lines=10)

    run_btn = gr.Button("⚡ Run DataFix Engine", variant="primary")

    with gr.Row():
        raw_preview = gr.Dataframe(label="❌ Before: Raw Corrupted Dataset")
        clean_preview = gr.Dataframe(label="✅ After: Repaired Clean Dataset")

    quarantine_preview = gr.Dataframe(label="⚠️ Quarantined Records (Isolated for Human Review)")

    with gr.Row():
        clean_download = gr.File(label="Download Repaired CSV")
        quarantine_download = gr.File(label="Download Quarantined CSV")

    run_btn.click(
        fn=run_datafix_pro,
        inputs=file_input,
        outputs=[audit_output, raw_preview, clean_preview, quarantine_preview, clean_download, quarantine_download]
    )

app.launch(share=True)

/tmp/ipykernel_852/2115883373.py:92: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://870995845a0a1f6541.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
